In [2]:
from boututils.datafile import DataFile
from boutdata.collect import collect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys, pathlib
import platform
import traceback
import xarray as xr
import xbout
from pathlib import Path
import xhermes as xh

sys.path.append(os.path.join(r"/users/jlb647/scratch/simulation_program/hermes-3_sim/sdtool_load_test/sdtools"))
sys.path.append(os.path.join(r"/users/jlb647/scratch/simulation_program/hermes-3_sim/analysis/my_notebooks/notebooks/hermes-3/transients"))
sys.path.append(os.path.join(r"/users/jlb647/scratch/simulation_program/hermes-3_sim/analysis/my_notebooks/notebooks/hermes-3/general_functions"))


from plotting_functions import *
from convergence_functions import * 

from hermes3.case_db import *
from hermes3.casedeck import*
from hermes3.load import *
from hermes3.named_selections import *
from hermes3.plotting import *
from hermes3.grid_fields import *
from hermes3.accessors import *
from hermes3.utils import *
from hermes3.fluxes import *
from hermes3.selectors import *
from hermes3.front_tracking import *
# from hermes3.balance1d import *

# plt.style.use('ggplot')
plt.rcParams.update({'font.size': 10})
linewidth = 3
markersize = 15



# plt.style.use('ggplot')
plt.style.use('default')
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["axes.linewidth"] = 1
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['axes.grid'] = True
plt.rcParams.update({'font.size': 16})



%load_ext autoreload
%autoreload 2


# I/O

In [4]:
parent_dir = '/users/jlb647/scratch/simulation_program/hermes-3_sim/simulation_dir/2025-04_wringing_bug'
grid_dir = os.path.join(parent_dir, 'Grid_resolution_scan')
sound_dir = os.path.join(parent_dir, 'Sound_speed_scan')


cs_grid = dict()
cs_sound = dict()

for i in ['ny_400', 'ny_800', 'ny_1200']:
    print(i)
    cs_grid[i] = Load.case_1D(os.path.join(grid_dir, i), use_squash=True, guard_replace=False).ds
 
for i in ['1.0x_fastest_wave_factor', '1.25x_fastest_wave_factor', '1.5x_fastest_wave_factor', '1.75x_fastest_wave_factor', '2.0x_fastest_wave_factor']:
    print(i)
    cs_sound[i] = Load.case_1D(os.path.join(sound_dir, i), use_squash=True, guard_replace=False).ds



# crash_case_beuler = Load.case_1D('/users/jlb647/scratch/simulation_program/hermes-3_sim/simulation_dir/2025-03_updated_glimmer/32_core_cold_start', use_squash=True, guard_replace=False).ds
# crash_case_cvode =  Load.case_1D('/users/jlb647/scratch/simulation_program/hermes-3_sim/simulation_dir/2025-03_updated_glimmer/32_core_cold_start_cvode', use_squash=True, guard_replace=False).ds
# att_datt = Load.case_1D('/users/jlb647/scratch/simulation_program/hermes-3_sim/simulation_dir/2025-04_updated_glimmer/scenario_development/Detached/case_04_ITER_format_03_cont', use_squash=True, guard_replace=False).ds
# working_case =  Load.case_1D('/users/jlb647/scratch/simulation_program/hermes-3_sim/simulation_dir/2025-04_updated_glimmer/scenario_development/Detached/case_07_cold_start_detached_20_cores', use_squash=True, guard_replace=False).ds

ny_400
- Looking for squash file
- Squash file found. squash date 04/29/2025, 16:18:43, dmp file date 04/28/2025, 01:48:48
Skipping unnormalisation
ny_800
- Looking for squash file
- Squash file found. squash date 04/29/2025, 16:18:58, dmp file date 04/28/2025, 01:43:40
Skipping unnormalisation
ny_1200
- Looking for squash file
- Squash file found. squash date 04/29/2025, 16:19:12, dmp file date 04/28/2025, 02:06:20
Skipping unnormalisation
1.0x_fastest_wave_factor
- Looking for squash file
- Squashoutput file not found, creating...
- Done
Skipping unnormalisation
1.25x_fastest_wave_factor
- Looking for squash file
- Squashoutput file not found, creating...
- Done
Skipping unnormalisation
1.5x_fastest_wave_factor
- Looking for squash file
- Squashoutput file not found, creating...
- Done
Skipping unnormalisation
1.75x_fastest_wave_factor
- Looking for squash file
- Squashoutput file not found, creating...
- Done
Skipping unnormalisation
2.0x_fastest_wave_factor
- Looking for squash fil

In [5]:
def detachment_front_index(ds, time=None):
    """
    Function to find the location of the detachment in the simulation
    """
    if time == None:
        ds = ds.isel(t=-1)

    else:
        ds = ds.isel(t=time)

    # Get the detachment location
    try:
        detachment_idx = np.where(ds['Te'][2:-2] < 5)[0][0]
    except:
        # If the detachment index is not found, return None
        print("No detachment found")
        detachment_idx = 0

    return detachment_idx

In [6]:
index = detachment_front_index(att_datt, time = 5)
print(index)

NameError: name 'att_datt' is not defined

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import imageio
import os


def detatchment_var(ds, var, time_range = (0, -1), save_gif = True, save_mp4 = True, static_scale = False, S_range = None, animation_name = "animation", output_dir = None, mp4_name = "animation.mp4"):
    import warnings
    warnings.filterwarnings("ignore", category=DeprecationWarning)
    frames = []
    if output_dir is None:
        output_dir = "/users/jlb647/scratch/simulation_program/hermes-3_sim/analysis/my_notebooks/notebooks/hermes-3/transients/1_D/2025_updates/gif_frames"
    else:
        output_dir = output_dir
    os.makedirs(output_dir, exist_ok=True)

    # Loop over time indices
    for i in np.linspace(time_range[0], len(ds['t'][time_range[0]:time_range[1]]) - 1, 100, dtype=int):
        print(f"Creating frame {i}")
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        ds_temp = ds.isel(t=i)
        
        det_idx = detachment_front_index(ds, time=i)
        
        ax.plot(ds_temp['y'][::-1], ds_temp[var], marker = 'o', markersize = 5)
        # ax.set_xbound(0, ds['y'][::-1][det_idx] + 3)
        ax.set_title(f"Time index: {i} \n name = {animation_name}")



        ax.set_xlabel('Distance from target (cm)')
        ax.set_ylabel(f'{var} ({ds[var].units})')

        if static_scale:
            ax.set_ybound(np.min(ds['NVd']) * 1.1,np.max(ds['NVd']) * 1.1)

        if S_range == None:
            # print(ds['y'].values[::-1][det_idx] + 3)
            
            ax.set_xbound(ds['y'][::-1][det_idx] + -3, ds['y'][::-1][det_idx] + 3)

        else:
            ax.set_xbound(0 , S_range)
        # ax.set_yscale('symlog')

        frame_path = f"{output_dir}/frame_{i:03d}.png"
        plt.savefig(frame_path)
        plt.close(fig)

        frames.append(imageio.imread(frame_path))

    # Save the gif
    if save_gif:
        print(f"Saving gif to {output_dir}/{animation_name}.gif")
        imageio.mimsave(f"{animation_name}.gif", frames, duration=0.8)

    # Read the gif using imageio
    gif_path = f'{animation_name}.gif'
    mp4_path = f'{mp4_name}.mp4'

    # Use imageio to convert gif to mp4
    with imageio.get_writer(mp4_path, format='mp4', fps=2) as writer:
        print(f"Converting {gif_path} to {mp4_path}")
        gif = imageio.mimread(gif_path)  # Read gif frames
        for frame in gif:
            # Convert to RGB (3 channels)
            frame_rgb = np.array(frame)
            
            # If the frame has 4 channels (RGBA), convert it to RGB (ignore the alpha channel)
            if frame_rgb.shape[-1] == 4:
                frame_rgb = frame_rgb[..., :3]
            
            # If it's grayscale (2D array), convert to 3 channels (RGB)
            elif frame_rgb.ndim == 2:
                frame_rgb = np.stack([frame_rgb] * 3, axis=-1)
            
            writer.append_data(frame_rgb)


In [14]:
# detatchment_var(att_datt, 'NVd', time_range=(0, -1), save_gif=True, save_mp4=True, static_scale=False, S_range=5, animation_name="animation_1", output_dir=None, mp4_name="animatio_1")


for key, value in cs_sound.items():
    print(key)
    detatchment_var(value, 'NVd', time_range=(0, -1), save_gif=True, save_mp4=True, static_scale=False, S_range=None, animation_name=f"./{key}", output_dir=None, mp4_name=f"./{key}")

for key, value in cs_grid.items():
    print(key)
    detatchment_var(value, 'NVd', time_range=(0, -1), save_gif=True, save_mp4=True, static_scale=False, S_range=None, animation_name=f"./{key}", output_dir=None, mp4_name=f"./{key}")

# for i in cs_grid.values():
#     print(i)
#     var_name = [name for name, value in globals().items() if value is i]
#     print(var_name[0] if var_name else "Unknown")
#     detatchment_var(i, 'NVd', time_range=(0, -1), save_gif=True, save_mp4=True, static_scale=True, S_range=None, animation_name=f"./{var_name[0]}", output_dir=None, mp4_name=f"./{var_name[0]}")

1.0x_fastest_wave_factor
Creating frame 0
No detachment found
Creating frame 0
No detachment found
Creating frame 1
Creating frame 1
Creating frame 2
Creating frame 3
Creating frame 3
Creating frame 4
Creating frame 5
Creating frame 5
Creating frame 6
Creating frame 7
Creating frame 7
Creating frame 8
Creating frame 8
Creating frame 9
Creating frame 10
Creating frame 10
Creating frame 11
Creating frame 12
Creating frame 12
Creating frame 13
Creating frame 14
Creating frame 14
Creating frame 15
Creating frame 15
Creating frame 16
Creating frame 17
Creating frame 17
Creating frame 18
Creating frame 19
Creating frame 19
Creating frame 20
Creating frame 21
Creating frame 21
Creating frame 22
Creating frame 22
Creating frame 23
Creating frame 24
Creating frame 24
Creating frame 25
Creating frame 26
Creating frame 26
Creating frame 27
Creating frame 28
Creating frame 28
Creating frame 29
Creating frame 29
Creating frame 30
Creating frame 31
Creating frame 31
Creating frame 32
Creating frame 

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1000, 500) to (1008, 512) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
[swscaler @ 0x5d1fa80] Warning: data is not aligned! This can lead to a speed loss


1.25x_fastest_wave_factor
Creating frame 0
No detachment found
Creating frame 0
No detachment found
Creating frame 1
Creating frame 2
Creating frame 2
Creating frame 3
Creating frame 4
Creating frame 4
Creating frame 5
Creating frame 6
Creating frame 6
Creating frame 7
Creating frame 8
Creating frame 8
Creating frame 9
Creating frame 10
Creating frame 10
Creating frame 11
Creating frame 12
Creating frame 13
Creating frame 13
Creating frame 14
Creating frame 15
Creating frame 15
Creating frame 16
Creating frame 17
Creating frame 17
Creating frame 18
Creating frame 19
Creating frame 19
Creating frame 20
Creating frame 21
Creating frame 21
Creating frame 22
Creating frame 23
Creating frame 24
Creating frame 24
Creating frame 25
Creating frame 26
Creating frame 26
Creating frame 27
Creating frame 28
Creating frame 28
Creating frame 29
Creating frame 30
Creating frame 30
Creating frame 31
Creating frame 32
Creating frame 32
Creating frame 33
Creating frame 34
Creating frame 35
Creating fram

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1000, 500) to (1008, 512) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
[swscaler @ 0x6261a80] Warning: data is not aligned! This can lead to a speed loss


1.5x_fastest_wave_factor
Creating frame 0
No detachment found
Creating frame 0
No detachment found
Creating frame 1
Creating frame 2
Creating frame 3
Creating frame 3
Creating frame 4
Creating frame 5
Creating frame 6
Creating frame 7
Creating frame 7
Creating frame 8
Creating frame 9
Creating frame 10
Creating frame 11
Creating frame 11
Creating frame 12
Creating frame 13
Creating frame 14
Creating frame 14
Creating frame 15
Creating frame 16
Creating frame 17
Creating frame 18
Creating frame 18
Creating frame 19
Creating frame 20
Creating frame 21
Creating frame 22
Creating frame 22
Creating frame 23
Creating frame 24
Creating frame 25
Creating frame 26
Creating frame 26
Creating frame 27
Creating frame 28
Creating frame 29
Creating frame 29
Creating frame 30
Creating frame 31
Creating frame 32
Creating frame 33
Creating frame 33
Creating frame 34
Creating frame 35
Creating frame 36
Creating frame 37
Creating frame 37
Creating frame 38
Creating frame 39
Creating frame 40
Creating fra

...

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1000, 500) to (1008, 512) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
[swscaler @ 0x6dad880] Warning: data is not aligned! This can lead to a speed loss


ny_1200
Creating frame 0
No detachment found
Creating frame 0
No detachment found
Creating frame 0
No detachment found
Creating frame 1
Creating frame 1
Creating frame 2
Creating frame 2
Creating frame 3
Creating frame 3
Creating frame 4
Creating frame 4
Creating frame 5
Creating frame 5
Creating frame 5
Creating frame 6
Creating frame 6
Creating frame 7
Creating frame 7
Creating frame 8
Creating frame 8
Creating frame 9
Creating frame 9
Creating frame 10
Creating frame 10
Creating frame 10
Creating frame 11
Creating frame 11
Creating frame 12
Creating frame 12
Creating frame 13
Creating frame 13
Creating frame 14
Creating frame 14
Creating frame 15
Creating frame 15
Creating frame 15
Creating frame 16
Creating frame 16
Creating frame 17
Creating frame 17
Creating frame 18
Creating frame 18
Creating frame 19
Creating frame 19
Creating frame 20
Creating frame 20
Creating frame 20
Creating frame 21
Creating frame 21
Creating frame 22
Creating frame 22
Creating frame 23
Creating frame 23


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1000, 500) to (1008, 512) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Converting ./ny_1200.gif to ./ny_1200.mp4


[swscaler @ 0x5b31880] Warning: data is not aligned! This can lead to a speed loss


In [125]:
detatchment_var(working_case, 'NVd', time_range=(0, -1), save_gif=True, save_mp4=True, static_scale=False, S_range=None, animation_name=f"./working_case", output_dir=None, mp4_name=f"./working_case")

Creating frame 0
No detachment found
85.40504796875001
Creating frame 3
12.74751296875
Creating frame 7
19.373010468749996
Creating frame 10
22.992977343749995
Creating frame 14
27.00554484375
Creating frame 18
30.247876093749998
Creating frame 21
32.46261921874999
Creating frame 25
35.09118046875
Creating frame 29
37.50423546874999
Creating frame 32
39.128962968749995
Creating frame 36
41.156438593749996
Creating frame 39
42.57696046874999
Creating frame 43
44.38916046874999
Creating frame 47
46.02736671874999
Creating frame 50
47.101054218749994
Creating frame 54
48.68391046874999
Creating frame 58
49.977588593749985
Creating frame 61
44.52694609374999
Creating frame 65
35.69997921875
Creating frame 68
36.45578859375
Creating frame 72
36.75649796875
Creating frame 76
38.39390984375
Creating frame 79
38.54138171875
Creating frame 83
39.42136984374999
Creating frame 87
40.00341609374998
Creating frame 90
40.43752921874999
Creating frame 94
41.013117968749995
Creating frame 97
41.442387

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1000, 500) to (1008, 512) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
[swscaler @ 0x6bc8880] Warning: data is not aligned! This can lead to a speed loss
